In [2]:
import pandas as pd
import numpy as np

def descrever_dados(df, coluna, coluna_grupo=None, remover_outliers=False):
    """
    Analisa uma coluna numérica de um DataFrame, calculando estatísticas descritivas
    com suporte a agrupamento por amostras. Identifica outliers via critério IQR
    (1.5 * IQR) e permite sua remoção.

    Parâmetros:
    -----------
    df : pandas.DataFrame
        O DataFrame contendo os dados.
    coluna : str
        Nome da coluna numérica a ser analisada.
    coluna_grupo : str, opcional
        Nome da coluna para agrupamento dos dados (ex: 'Amostra', 'Categoria').
    remover_outliers : bool, default False
        Se True, retorna a tabela descritiva e uma cópia do DataFrame sem os outliers.

    Retorna:
    --------
    tabela : pandas.DataFrame
        Tabela descritiva formatada.
    df_limpo : pandas.DataFrame (apenas se remover_outliers=True)
        DataFrame filtrado sem os outliers.
    """
    if coluna not in df.columns:
        raise ValueError(f"A coluna '{coluna}' não foi encontrada no DataFrame.")

    # Função interna para calcular as estatísticas de cada amostra/conjunto
    def calcular_estatisticas(s):
        s_clean = s.dropna()
        n = len(s_clean)
        if n == 0:
            return pd.Series({
                'Total': 0, 'Média': np.nan, 'Mediana': np.nan, 'Desvio Padrão': np.nan,
                'Q1 (25%)': np.nan, 'Q3 (75%)': np.nan, 'IQR': np.nan,
                'Outliers (Qtd)': 0, 'Outliers (Valores)': []
            })

        media = s_clean.mean()
        mediana = s_clean.median()
        std = s_clean.std(ddof=1)  # Desvio padrão amostral

        q1 = s_clean.quantile(0.25)
        q3 = s_clean.quantile(0.75)
        iqr = q3 - q1

        lim_inferior = q1 - 1.5 * iqr
        lim_superior = q3 + 1.5 * iqr

        outliers = s_clean[(s_clean < lim_inferior) | (s_clean > lim_superior)].tolist()

        return pd.Series({
            'Total': n,
            'Média': round(media, 4),
            'Mediana': round(mediana, 4),
            'Desvio Padrão': round(std, 4),
            'Q1 (25%)': round(q1, 4),
            'Q3 (75%)': round(q3, 4),
            'IQR': round(iqr, 4),
            'Outliers (Qtd)': len(outliers),
            'Outliers (Valores)': outliers if len(outliers) > 0 else 'Nenhum'
        })

    # 1. Agrupamento por amostras (se informado)
    if coluna_grupo:
        if coluna_grupo not in df.columns:
            raise ValueError(f"A coluna de agrupamento '{coluna_grupo}' não foi encontrada.")

        tabela = df.groupby(coluna_grupo)[coluna].apply(calcular_estatisticas).unstack()
        print(f"📊 --- RESUMO ESTATÍSTICO AGRUPADO POR '{coluna_grupo}' ---")
    else:
        tabela = pd.DataFrame([calcular_estatisticas(df[coluna])], index=['Geral'])
        print(f"📊 --- RESUMO ESTATÍSTICO DA COLUNA '{coluna}' ---")

    # Exibição da Tabela
    display(tabela)

    # 2. Identificação Global de Outliers
    q1_g = df[coluna].quantile(0.25)
    q3_g = df[coluna].quantile(0.75)
    iqr_g = q3_g - q1_g
    lim_inf_g = q1_g - 1.5 * iqr_g
    lim_sup_g = q3_g + 1.5 * iqr_g

    mask_outliers = (df[coluna] < lim_inf_g) | (df[coluna] > lim_sup_g)
    outliers_df = df[mask_outliers]

    print("\n🚨 --- DETECÇÃO E REMOÇÃO DE OUTLIERS ---")
    if len(outliers_df) > 0:
        print(f"Foram encontrados {len(outliers_df)} outlier(s) na coluna '{coluna}':")
        print(f"• Limites válidos (IQR): Limite Inferior = {lim_inf_g:.4f} | Limite Superior = {lim_sup_g:.4f}\n")
        display(outliers_df[[coluna_grupo, coluna]] if coluna_grupo else outliers_df[[coluna]])
    else:
        print(f"✅ Nenhum outlier foi identificado na coluna '{coluna}'.")

    # 3. Remoção de Outliers
    if remover_outliers:
        df_limpo = df[~mask_outliers].copy()
        print(f"\n🧹 Outliers removidos! O DataFrame limpo foi retornado ({len(df_limpo)} linhas restantes).")
        return tabela, df_limpo

    return tabela